# Kaggle Titanic - leaderboard attempt

going back to titanic with a more competitive solution. target: top 25%.


plan:
- proper preprocessing pipeline
- title extraction from names
- family size feature
- cabin letter extraction
- xgboost + voting ensemble
- 5-fold cv to estimate leaderboard score


In [7]:
import pandas as pd
import numpy as np
import os

# expects data/titanic/{train,test}.csv from the kaggle download
train = pd.read_csv('data/titanic/train.csv')
test = pd.read_csv('data/titanic/test.csv')
print(train.shape, test.shape)


In [8]:
train.head()


In [9]:
# title from name
import re
def title_of(name):
    m = re.search(r',\s*([^.]+)\.', name)
    return m.group(1).strip() if m else 'Unknown'
train['Title'] = train['Name'].apply(title_of)
test['Title'] = test['Name'].apply(title_of)
train['Title'].value_counts()


In [10]:
# family size
for df in [train, test]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)


In [11]:
# cabin letter
for df in [train, test]:
    df['CabinLetter'] = df['Cabin'].fillna('U').str[0]
train['CabinLetter'].value_counts()


In [12]:
# fare per person
for df in [train, test]:
    df['FarePer'] = df['Fare'] / df['FamilySize']


In [13]:
# fillna age by title group
age_by_title = train.groupby('Title')['Age'].median()
for df in [train, test]:
    df['Age'] = df.apply(lambda r: r['Age'] if pd.notna(r['Age']) else age_by_title.get(r['Title'], train['Age'].median()), axis=1)


In [14]:
# encode
from sklearn.preprocessing import LabelEncoder
for c in ['Sex','Embarked','Title','CabinLetter']:
    le = LabelEncoder()
    combined = pd.concat([train[c].astype(str), test[c].astype(str)])
    le.fit(combined)
    train[c] = le.transform(train[c].astype(str))
    test[c] = le.transform(test[c].astype(str))


In [15]:
# xgboost baseline
import xgboost as xgb
from sklearn.model_selection import cross_val_score
feats = ['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked','Title','FamilySize','IsAlone','CabinLetter','FarePer']
clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1)
scores = cross_val_score(clf, train[feats], train['Survived'], cv=5, scoring='accuracy')
print(scores.mean(), scores.std())


In [16]:
# voting ensemble
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
v = VotingClassifier([
    ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=4)),
    ('rf', RandomForestClassifier(n_estimators=300)),
    ('gb', GradientBoostingClassifier(n_estimators=200)),
    ('lr', LogisticRegression(max_iter=1000)),
], voting='soft')
scores = cross_val_score(v, train[feats], train['Survived'], cv=5)
print(scores.mean())


In [17]:
# write submission
v.fit(train[feats], train['Survived'])
sub = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': v.predict(test[feats])})
sub.to_csv('submission.csv', index=False)
sub.head()


In [18]:
# leaderboard score around 0.79 with this. pretty close to top-25%.


### leaderboard score: 0.78947. close to top-25%.


### still improving. trying with different ensemble weights next.


### closed at 0.79 on public lb, didn't improve. moving on.


swapped to mlflow autolog. fewer lines, same artifacts.

added gradient clipping at 1.0, training stabilized.

In [ ]:
# print param count
# from utils import count_params
# print(count_params(model))


note: one more pass
